# 04 Final Forecast Construction


This notebook recombines the CV branch and the service-metric branch into the final August interval submission.

Before post-processing, the forecast preserves the completed daily anchors. The final competition submission then applies the v42 conservative post-processing bias: +5% to call volume and +3% to CCT, followed by nonnegativity, rate, and count guardrails.

## Core Implementation Used Here

The cells below call `src/pipeline.py` so the notebooks and command-line runner stay consistent. For reviewability, this section shows the exact source code for the functions used in this notebook.

```python
def combine_forecast(cv_forecast: pd.DataFrame, service_forecast: pd.DataFrame, template: pd.DataFrame, apply_bias: bool = True) -> pd.DataFrame:
    merged = cv_forecast.merge(service_forecast, on=["Portfolio", "Date", "slot"], how="inner")
    if apply_bias:
        merged["interval_cv"] = merged["interval_cv"] * CV_BIAS
        merged["interval_cct"] = merged["interval_cct"] * CCT_BIAS
    merged["interval_cv"] = np.clip(merged["interval_cv"], 0, None)
    merged["interval_cct"] = np.clip(merged["interval_cct"], 0, None)
    merged["interval_abandoned_calls"] = np.clip(merged["interval_abandoned_calls"], 0, merged["interval_cv"])
    merged["interval_ar"] = np.where(merged["interval_cv"] > 0, merged["interval_abandoned_calls"] / merged["interval_cv"], 0.0)
    merged["interval_ar"] = np.clip(merged["interval_ar"], 0, 1)
    merged["Calls_Offered"] = np.round(merged["interval_cv"]).astype(int)
    merged["Abandoned_Calls"] = np.round(merged["interval_abandoned_calls"]).astype(int)

    rows = []
    august = pd.date_range(AUGUST_START, AUGUST_END, freq="D")
    for day_idx, dt in enumerate(august, start=1):
        for slot in range(SLOTS_PER_DAY):
            row = {"Month": "August", "Day": str(day_idx), "Interval": slot_label(slot)}
            for portfolio in PORTFOLIOS:
                rec = merged[(merged["Portfolio"] == portfolio) & (merged["Date"] == dt) & (merged["slot"] == slot)]
                if rec.empty:
                    raise ValueError(f"Missing forecast row for {portfolio} {dt.date()} slot {slot}")
                r = rec.iloc[0]
                calls = int(r["Calls_Offered"])
                abandoned = min(int(r["Abandoned_Calls"]), calls)
                row[f"Calls_Offered_{portfolio}"] = calls
                row[f"Abandoned_Calls_{portfolio}"] = abandoned
                row[f"Abandoned_Rate_{portfolio}"] = round(float(abandoned / calls if calls > 0 else 0.0), 6)
                row[f"CCT_{portfolio}"] = round(float(r["interval_cct"]), 2)
            rows.append(row)

    submission = pd.DataFrame(rows)[template.columns.tolist()]
    validate_submission(submission)
    return submission

def validate_submission(submission: pd.DataFrame) -> None:
    if submission.shape != (31 * SLOTS_PER_DAY, 19):
        raise AssertionError(f"Unexpected submission shape: {submission.shape}")
    if submission.isnull().any().any():
        raise AssertionError("Submission contains null values.")
    for portfolio in PORTFOLIOS:
        if (submission[f"Calls_Offered_{portfolio}"] < 0).any():
            raise AssertionError(f"Negative call volume in portfolio {portfolio}.")
        if (submission[f"Abandoned_Calls_{portfolio}"] < 0).any():
            raise AssertionError(f"Negative abandoned calls in portfolio {portfolio}.")
        if (submission[f"Abandoned_Calls_{portfolio}"] > submission[f"Calls_Offered_{portfolio}"]).any():
            raise AssertionError(f"Abandoned calls exceed offered calls in portfolio {portfolio}.")
        if not submission[f"Abandoned_Rate_{portfolio}"].between(0, 1).all():
            raise AssertionError(f"Abandon rate outside [0, 1] in portfolio {portfolio}.")
        if (submission[f"CCT_{portfolio}"] < 0).any():
            raise AssertionError(f"Negative CCT in portfolio {portfolio}.")

def run_final_submission(apply_bias: bool = True) -> pd.DataFrame:
    ensure_dirs()
    raw = load_raw_data()
    cv_forecast = pd.read_csv(OUTPUT_DIR / "cv_interval_forecast_unbiased.csv", parse_dates=["Date"])
    service_forecast = pd.read_csv(OUTPUT_DIR / "service_interval_forecast_unbiased.csv", parse_dates=["Date"])
    submission = combine_forecast(cv_forecast, service_forecast, raw.template, apply_bias=apply_bias)
    output_name = "forecast_v42.csv" if apply_bias else "forecast_v42_pre_bias.csv"
    submission.to_csv(OUTPUT_DIR / output_name, index=False)
    return submission
```


In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'pipeline.py').exists():
        ROOT = candidate
        break
    if (candidate / 'datathon_final' / 'src' / 'pipeline.py').exists():
        ROOT = candidate / 'datathon_final'
        break
if ROOT is None:
    raise RuntimeError('Could not find datathon_final project root.')
sys.path.insert(0, str(ROOT))
from src import pipeline


## Construct submission files

`forecast_v42_pre_bias.csv` is the anchor-reconciled forecast before the competition bias. `forecast_v42.csv` is the final submission file with v42 post-processing applied.

In [ ]:
pre_bias = pipeline.run_final_submission(apply_bias=False)
final_submission = pipeline.run_final_submission(apply_bias=True)
print(f'Pre-bias rows: {len(pre_bias):,}')
print(f'Final rows: {len(final_submission):,}')
final_submission.head()

## Sanity checks

The final submission matches the template shape, contains no nulls, keeps counts nonnegative, keeps abandoned calls below offered calls, and keeps abandon rates in `[0, 1]`.

In [ ]:
pipeline.validate_submission(final_submission)
for p in pipeline.PORTFOLIOS:
    print(
        p,
        'calls=', int(final_submission[f'Calls_Offered_{p}'].sum()),
        'abandoned=', int(final_submission[f'Abandoned_Calls_{p}'].sum()),
        'avg_cct=', round(final_submission[f'CCT_{p}'].mean(), 2),
    )